# 04 - Evaluation Scores Analysis (EDA)

## Overview
This notebook provides a comprehensive analysis of VLM evaluation scores, including:
- **Glider Scores**: Automated evaluation scores.
- **VLM Judge Scores**: Scores from a VLM serving as a judge (e.g., Llama-4-Scout/Molmo).
- **Confidence**: Model confidence calibration.
- **Performance**: Analysis by task, split, and ground truth type.

## Goal
To generate publication-quality visualizations and insights for the VLM Router benchmark.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Setup paths
NOTEBOOK_DIR = os.getcwd()
ARTEMIS_DIR = os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(ARTEMIS_DIR)

from ares.db.connection import get_engine
from ares.configs.db_config import DB_CONNECTION_STRING

# Plotting settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Loading
Load data from the evaluation database. We attempt to use the `vlm_full_data` view if available, otherwise we join the tables manually.

In [ ]:
engine = get_engine()

try:
    # Try fetching from the full view first
    print("Attempting to load data from vlm_full_data view...")
    df = pd.read_sql("SELECT * FROM vlm_full_data", engine)
    print(f"Successfully loaded {len(df)} rows from vlm_full_data.")
except Exception as e:
    print(f"View lookup failed ({e}), falling back to manual join...")
    # Manual join fallback
    query = """
    SELECT 
        s.sample_id, s.router_task, s.data_split, s.ground_truth_type,
        r.model_name, r.is_correct, r.confidence_score, r.score_f1, r.score_exact_match,
        r.latency_ms, r.input_tokens, r.output_tokens,
        e.glider_score, e.glider_reasoning
    FROM vlm_samples s
    JOIN vlm_responses r ON s.sample_id = r.sample_id
    LEFT JOIN vlm_evaluations e ON r.response_id = e.response_id
    """
    df = pd.read_sql(query, engine)
    print(f"Loaded {len(df)} rows via manual join.")
    
# Check for VLM Judge columns (handling name ambiguity)
judge_cols = [c for c in df.columns if 'judge' in c.lower() or 'molmo' in c.lower()]
if judge_cols:
    print(f"Found VLM Judge columns: {judge_cols}")
else:
    print("No specific VLM Judge columns found. Will proceed with Glider and standard metrics.")
    
# Preprocessing
df['glider_score'] = pd.to_numeric(df['glider_score'], errors='coerce')
df['confidence_score'] = pd.to_numeric(df['confidence_score'], errors='coerce')
df.head()

## 2. Evaluation Coverage Overview

In [ ]:
# Coverage by Model
coverage = df.groupby('model_name').agg({
    'sample_id': 'count',
    'glider_score': 'count',
    'is_correct': 'count'
}).rename(columns={
    'sample_id': 'Total Responses',
    'glider_score': 'Glider Eval Count',
    'is_correct': 'Exact Match Count'
})
coverage['Glider Coverage %'] = (coverage['Glider Eval Count'] / coverage['Total Responses'] * 100).round(1)
display(coverage)

# Visualization: Coverage Heatmap (Task vs Model)
plt.figure(figsize=(14, 8))
task_model_counts = df.pivot_table(index='router_task', columns='model_name', values='glider_score', aggfunc='count')
sns.heatmap(task_model_counts, annot=True, fmt='g', cmap='viridis', cbar_kws={'label': 'Count of Glider Scores'})
plt.title('Glider Evaluation Coverage: Task vs Model')
plt.tight_layout()
plt.show()

## 3. Glider Score Analysis

In [ ]:
# 3. Glider Score Analysis
# Distribution
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='glider_score', hue='model_name', multiple='stack', bins=6, binrange=(0, 5))
plt.title('Distribution of Glider Scores by Model')
plt.xlabel('Glider Score (0-5)')
plt.show()

# Boxplot Comparison
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='model_name', y='glider_score')
plt.title('Glider Score Distribution per Model')
plt.xticks(rotation=45)
plt.show()

# Summary Stats
glider_stats = df.groupby('model_name')['glider_score'].describe().sort_values('mean', ascending=False)
display(glider_stats)

## 4. VLM Judge Score Analysis

In [ ]:
# 4. VLM Judge Score Analysis (Conditional)
if judge_cols:
    judge_score_col = next((c for c in judge_cols if 'score' in c), judge_cols[0])
    print(f"Analyzing VLM Judge Score Column: {judge_score_col}")
    
    # Ensure numeric
    df[judge_score_col] = pd.to_numeric(df[judge_score_col], errors='coerce')

    # Distribution
    plt.figure(figsize=(12, 6))
    sns.histplot(data=df, x=judge_score_col, hue='model_name', multiple='stack', bins=10)
    plt.title(f'Distribution of {judge_score_col}')
    plt.show()
    
    # Correlation with Glider
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=df, x='glider_score', y=judge_score_col, alpha=0.3)
    plt.title('Glider Score vs VLM Judge Score')
    plt.show()
else:
    print("No VLM Judge score columns found. Skipping judge-specific analysis.")

## 5. Performance by Task

In [ ]:
# 5. Performance by Task
task_perf = df.groupby(['router_task', 'model_name'])['glider_score'].mean().unstack()

# Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(task_perf, annot=True, fmt='.2f', cmap='RdYlGn', center=2.5)
plt.title('Average Glider Score: Task vs Model')
plt.show()

# Best Model per Task
best_models = task_perf.idxmax(axis=1).reset_index(name='Best Model')
best_models['Max Score'] = task_perf.max(axis=1).values
display(best_models)

## 6. Performance by Ground Truth Type & Split

In [ ]:
# 6. Ground Truth Type & Split
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# By GT Type
sns.barplot(data=df, x='ground_truth_type', y='glider_score', hue='model_name', ax=axes[0])
axes[0].set_title('Glider Score by Ground Truth Type')
axes[0].tick_params(axis='x', rotation=45)

# By Data Split
sns.barplot(data=df, x='data_split', y='glider_score', hue='model_name', ax=axes[1])
axes[1].set_title('Glider Score by Data Split')

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# 7. Correlation Analysis
corr_cols = ['glider_score', 'confidence_score', 'score_f1', 'latency_ms', 'input_tokens']
if judge_cols:
    corr_cols.append(judge_score_col)

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation Matrix of Metrics')
plt.show()

## 8. Confidence Calibration

In [ ]:
# 8. Confidence Calibration
# Bin confidence scores and calculate accuracy
df_conf = df.dropna(subset=['confidence_score', 'is_correct']).copy()
df_conf['conf_bin'] = pd.cut(df_conf['confidence_score'], bins=np.linspace(0, 1, 11), labels=np.linspace(0.05, 0.95, 10))

calibration = df_conf.groupby('conf_bin')['is_correct'].mean().reset_index()
calibration['conf_bin'] = calibration['conf_bin'].astype(float)

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
plt.plot(calibration['conf_bin'], calibration['is_correct'], 'o-', label='Model Performance')
plt.xlabel('Confidence Score')
plt.ylabel('Accuracy (is_correct)')
plt.title('Reliability Diagram (Calibration Curve)')
plt.legend()
plt.show()

# ECE Calculation (proxy)
ece = np.abs(calibration['conf_bin'] - calibration['is_correct']).mean()
print(f"Estimated Calibration Error (ECE): {ece:.4f}")

## 9. Evaluation Reasoning Examples

In [ ]:
# 9. Qualitative Examples
# Sample high and low score examples
print("--- High Quality Responses (Glider = 5) ---")
high_quality = df[df['glider_score'] == 5].sample(3) if not df[df['glider_score'] == 5].empty else df.head(3)
for _, row in high_quality.iterrows():
    print(f"Sample: {row['sample_id']} | Model: {row['model_name']}")
    print(f"Reasoning: {row['glider_reasoning']}")
    print("-" * 50)

print("\n--- Low Quality Responses (Glider <= 1) ---")
low_quality = df[df['glider_score'] <= 1].sample(3) if not df[df['glider_score'] <= 1].empty else df.head(3)
for _, row in low_quality.iterrows():
    print(f"Sample: {row['sample_id']} | Model: {row['model_name']}")
    print(f"Reasoning: {row['glider_reasoning']}")
    print("-" * 50)

## 10. Model Agreement & Consensus
Analyzing how often models agree on the *same sample*.

In [ ]:
# 10. Agreement
# Pivot sample x model (Glider Score)
pivot_scores = df.pivot_table(index='sample_id', columns='model_name', values='glider_score')

# Correlation between models (Do they rank samples similarly?)
plt.figure(figsize=(10, 8))
sns.heatmap(pivot_scores.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Model Agreement (Correlation of Glider Scores)')
plt.show()

## 11. Quality Drivers

In [ ]:
# 11. Quality Drivers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Latency vs Score
sns.scatterplot(data=df, x='latency_ms', y='glider_score', alpha=0.3, ax=axes[0])
axes[0].set_title('Latency vs Glider Score')
axes[0].set_xscale('log')

# Input Tokens vs Score
sns.scatterplot(data=df, x='input_tokens', y='glider_score', alpha=0.3, ax=axes[1])
axes[1].set_title('Token Count vs Glider Score')

plt.show()

## 12. Final Summary

In [ ]:
# 12. Final Summary Table
summary = df.groupby('model_name').agg({
    'glider_score': 'mean',
    'confidence_score': 'mean',
    'score_f1': 'mean',
    'score_exact_match': 'mean',
    'latency_ms': 'median',
    'estimated_cost_usd': 'sum'
}).sort_values('glider_score', ascending=False)

summary['cost_per_1k_samples'] = (summary['estimated_cost_usd'] / coverage['Total Responses'] * 1000).round(4)
display(summary)